# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`

This notebook demonstrates how to load and explore the FAIR^2 dataset using the `mlcroissant` library, referencing entities by their `@id` according to the Croissant standard.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Print general dataset info
print(f"{metadata.name}: {metadata.description}")
print(f"Identifier: {metadata.identifier}")
print(f"License: {metadata.license}")
print(f"Temporal Coverage: {metadata.temporal_coverage}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

In [ ]:
# List all record sets with their @id
print('Record Sets:')
record_sets = list(dataset.record_sets)
for rs in record_sets:
    print(f" - Name: {rs.name}, @id: {rs.id}")

# For each record set, show its fields and their @ids
for rs in record_sets:
    print(f"\nRecordSet: {rs.name} (@id: {rs.id})")
    for field in rs.fields:
        print(f"   Field: {field.name}, @id: {field.id}, Data type: {field.data_type}")

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. All references to record sets and fields use their `@id`s as shown above.

In [ ]:
# Create a list of record set @ids for extraction
record_sets_ids = [rs.id for rs in list(dataset.record_sets)]
dataframes = {}

for record_set_id in record_sets_ids:
    print(f"\nExtracting from RecordSet @id: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded DataFrame for {record_set_id}: columns={df.columns.tolist()} | shape={df.shape}")
        display(df.head())
    else:
        print(f"No records found in {record_set_id}.")

# For demonstration, select the first record set for subsequent EDA
main_record_set_id = None
if dataframes:
    main_record_set_id = list(dataframes.keys())[0]
    print(f"\nUsing main record set: {main_record_set_id}")
    print(f"Columns: {dataframes[main_record_set_id].columns.tolist()}")
    display(dataframes[main_record_set_id].head())
else:
    print('No dataframes were loaded. Please check the dataset schema contents.')

## 4. Exploratory Data Analysis (EDA)
Apply some basic processing: filter, normalize, and group by fields, referencing columns by their Croissant `@id`.

Replace the field IDs below with appropriate field `@id`s from your overview above.

In [ ]:
# Make sure there's data to analyze
if main_record_set_id is not None:
    df = dataframes[main_record_set_id].copy()
    print(f"\nSample of the records: {df.shape[0]} rows, Columns: {list(df.columns)}")
    
    # Try to pick a numeric field by inspecting data types
    numeric_field_id = None
    for col in df.columns:
        try:
            # Test for numeric columns
            if pd.api.types.is_numeric_dtype(df[col]):
                numeric_field_id = col
                break
        except Exception:
            continue
    print(f"Using numeric field @id: {numeric_field_id}")

    if numeric_field_id is not None:
        threshold = df[numeric_field_id].dropna().mean()
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f} (mean):")
        display(filtered_df.head())

        # Normalize this field (z-score)
        filtered_df[f"{numeric_field_id}_normalized"] = (
            (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) /
            filtered_df[numeric_field_id].std()
        )
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Try grouping by a categorical column
        group_field_id = None
        for col in df.columns:
            if col != numeric_field_id:
                # Try string/object but not special Croissant columns
                if pd.api.types.is_string_dtype(df[col]) and not col.startswith('_'):
                    group_field_id = col
                    break
        print(f"Group by field @id: {group_field_id}")
        if group_field_id is not None:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"Grouped statistics of {numeric_field_id} by {group_field_id}:")
            display(grouped_df.head())
    else:
        print("No numeric field found for EDA in the dataframe.")
else:
    print("No dataframe to analyze. Populate the dataframes dictionary first.")

## 5. Visualization
Visualize the distribution of a numeric field, or show group-wise means with a bar plot.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id is not None and numeric_field_id is not None:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.show()

    # If group_field_id/grouped_df exists, plot group means
    if 'grouped_df' in locals() and grouped_df is not None and group_field_id is not None:
        plt.figure(figsize=(10,4))
        sns.barplot(x=group_field_id, y=numeric_field_id, data=grouped_df)
        plt.title(f"Mean {numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print('Visualization skipped: insufficient data or no numeric field found.')

## 6. Conclusion
In this notebook, we loaded, explored, and visualized the FAIR^2 dataset using `mlcroissant`. All steps referenced Croissant entities by their `@id`, ensuring schema-driven and reproducible workflow. For further analysis, consult the dataset metadata and field definitions for precise `@id` mappings and deeper exploration.